# Release events — every TMDb release date, every country, every film

Thin driver for `release_events.py`. The module does the work and can run for
hours from a terminal; this notebook is for running stages with limits, reading
the coverage ledger, and looking at individual films.

| stage | what | cost at ~380k films |
|---|---|---|
| `universe` | crosswalk → `films` (one row per `tmdb_id`, tconsts joined) | seconds |
| `fetch` | one `append_to_response=release_dates,external_ids` call per film, verbatim payload stored | **~4–5 h at 25 rps**; resumable |
| `ledger` | raw → `events`: all six types, every country, note parsed and classified | ~2 min |
| `registry` | 40-festival verified seed + Wikidata (labels, aliases in 14 languages, country, usual month) | ~10–20 min, cached |
| `resolve` | evidence-scored festival resolution → `resolutions`, `appearances` | ~5 min |
| `windows` | per film × country premiere / theatrical / digital / physical / TV dates + gaps | ~2 min |
| `export` | CSVs incl. a `01_events`-shaped compat file | ~3 min |
| `credits` | `/movie/{id}/credits` per film → TMDb cast + crew | ~4 h |
| `metadata` | TMDb payload + credits + `imdb.db` → `film_metadata` / `films.csv` | ~10 min |
| `box_office.py` | Box Office Mojo grosses, territories, weekly US, weekend UK, matched to events | scoped; hours–days |
| `enrich.py` | Wikidata awards/nominations/distributors/ids; OMDb + Rotten Tomatoes ratings; TMDb awards pages | 1 h · daily · days · weeks |

**Key:** `export TMDB_API_KEY=...` (v3) or `TMDB_READ_TOKEN=...` (v4) *before* starting Jupyter.
Nothing here reads the key from a cell, and nothing writes it to disk.

Run top to bottom. Every stage is incremental: re-running a cell only does what is not done yet.

In [ ]:
# --- setup ------------------------------------------------------------------
import os, sys, sqlite3
from pathlib import Path

HERE = Path.cwd()
for cand in (HERE, *HERE.parents):
    if (cand / "release_events.py").exists():
        sys.path.insert(0, str(cand)); break
    if (cand / "release_events" / "release_events.py").exists():
        sys.path.insert(0, str(cand / "release_events")); break

import release_events as R
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

cfg = R.Config.from_env()
con = R.connect(cfg)
print(f"module     : v{R.__version__}  ({Path(R.__file__)})")
print(f"project    : {cfg.root}")
print(f"database   : {cfg.db_path}")
print(f"crosswalk  : {cfg.crosswalk_csv}  {'(found)' if cfg.crosswalk_csv.exists() else '(MISSING)'}")
print(f"mode       : {'LIVE (key in environment)' if cfg.live else 'OFFLINE - no TMDB_API_KEY / TMDB_READ_TOKEN set'}")

# Flip these on deliberately. The full fetch is hours; the registry refresh hits Wikidata.
RUN_FULL_FETCH = False
RUN_REGISTRY_REFRESH = False

## 1. Self-test

Offline, no key, no network: 158 checks over the parser, the note classifier, the
IDF-weighted resolver, the ledger and windows on the **real** Parasite / Monster
payloads, plus the live transport path against a fault-injecting fake TMDb and
fake Wikidata (429 with `Retry-After` as integer and HTTP-date, 5xx, 404, 401,
non-JSON, wrong id, dead socket). 27 of 30 deliberate mutations are caught;
the three that are not are provably inert.

In [ ]:
passed, total = R.selftest(cfg)
assert passed == total, f"{total - passed} self-test failures - do not run the pipeline on this build"

## 2. Universe

One film per `tmdb_id`. The crosswalk maps 378,876 tconsts to 378,721 distinct
TMDb ids: 154 ids carry two tconsts (IMDb duplicates, or one film with two IMDb
records) and are fetched once. Rows with no `tmdb_id` are counted, not stored as 0.

In [ ]:
if cfg.crosswalk_csv.exists():
    R.load_universe(cfg, con)
else:
    print("crosswalk not found - nothing loaded")

## 3. Fetch

Start with a small limit and **read the output**: the ok / gone / error split, the
rate, and the coverage ledger below. Then set `RUN_FULL_FETCH = True` in the
setup cell and re-run this cell — or, better, run it from a terminal so a closed
laptop does not stop it:

```bash
cd ADP_Project/release_events && python release_events.py fetch
```

Both paths write to the same database; Ctrl-C at any point loses nothing.

In [ ]:
if not cfg.live:
    print("no key in the environment - skipping fetch (set TMDB_API_KEY and restart the kernel)")
elif RUN_FULL_FETCH:
    R.fetch_stage(cfg, con)
else:
    R.fetch_stage(cfg, con, limit=300)

pd.read_sql("SELECT status, COUNT(*) n FROM films GROUP BY status", con)

## 4. Ledger and the coverage ledger

Pure function over the stored payloads; re-run it after any parser change
(`LEDGER_VERSION` bumps trigger a rebuild automatically). The fill-rate column
bounds everything downstream: a type-1 entry with an empty note is
indistinguishable from a city premiere.

In [ ]:
R.ledger_stage(cfg, con)
R.print_coverage(con)

## 5. Festival registry

The seed is the notebook's 40 festivals with verified Q-ids and edition windows.
`refresh_registry` adds every film festival Wikidata knows — thousands of items,
names in 14 languages — and never overwrites a seed value. Edition windows are
fetched lazily in `resolve`, only for festivals that notes actually name.

In [ ]:
if RUN_REGISTRY_REFRESH:
    R.refresh_registry(cfg, con)
else:
    R.seed_registry(cfg, con)
reg = pd.DataFrame(R.registry_rows(con).values())
reg["n_aliases"] = reg.aliases.str.len(); reg["n_editions"] = reg.editions.str.len()
print(f"{len(reg)} festivals; {reg.n_editions.sum()} editions; sources: {reg.source.value_counts().to_dict()}")
reg[["qid", "label", "country", "month", "n_aliases", "n_editions", "source"]].head(12)

## 6. Resolve

Every non-empty note is scored on five signals — name (IDF-weighted token F2),
country vs P17, date vs the edition window, usual month, section — and banded
accept / review / reject with its evidence. Accepted rows are deduplicated on
*(film, festival, edition)* into `appearances`.

In [ ]:
R.resolve_stage(cfg, con, fetch_editions_live=cfg.live)
R.report(cfg, con)

## 7. Windows and export

In [ ]:
R.windows_stage(cfg, con)
R.export_stage(cfg, con, compat=True)

## 8. Look at a film

`show()` takes a `tmdb_id` or a `tconst` and prints its events with the
resolution beside each one, its festival appearances, and its release windows.

In [ ]:
def show(key):
    if isinstance(key, str) and key.startswith("tt"):
        row = con.execute("SELECT tmdb_id FROM films WHERE '|'||tconsts||'|' LIKE ?", (f"%|{key}|%",)).fetchone()
        if not row:
            print(f"{key} not in the universe"); return
        key = row[0]
    f = con.execute("SELECT tmdb_id, tconsts, title, status, n_countries, n_entries FROM films WHERE tmdb_id=?", (key,)).fetchone()
    if f is None:
        print(f"tmdb_id {key} not in the universe (nothing fetched yet?)"); return
    print(f)
    ev = pd.read_sql("""
        SELECT e.country, e.type, e.type_name, e.date, e.note_raw, e.note_class,
               r.fest_label, r.score, r.band, r.edition_num, r.reason
        FROM events e LEFT JOIN resolutions r
          ON r.tmdb_id=e.tmdb_id AND r.country=e.country AND r.src_idx=e.src_idx
        WHERE e.tmdb_id=? ORDER BY e.date, e.country""", con, params=(key,))
    display(ev)
    display(pd.read_sql("SELECT fest_label, date, edition_num, types, countries, best_score, section "
                        "FROM appearances WHERE tmdb_id=? ORDER BY date", con, params=(key,)))
    display(pd.read_sql("SELECT * FROM windows WHERE tmdb_id=? ORDER BY country", con, params=(key,)))

show(496243)      # Parasite

## 9. What needs a human

`review_queue.csv` holds review-band rows that are not duplicates of an accepted
appearance. `unlisted_festival_notes.csv` is the festival-shaped tail that
matched nothing, by frequency — the promotion list: the top rows are the
festivals worth adding to the registry (usually by adding a Wikidata alias).

In [ ]:
rq = pd.read_sql("SELECT r.tmdb_id, f.title, r.country, e.date, e.note_raw, r.fest_label, r.score, r.reason "
                 "FROM resolutions r JOIN events e USING(tmdb_id, country, src_idx) JOIN films f USING(tmdb_id) "
                 "WHERE r.band='review' ORDER BY r.score DESC LIMIT 25", con)
display(rq)
p = cfg.out_dir / "unlisted_festival_notes.csv"
display(pd.read_csv(p).head(25) if p.exists() and p.stat().st_size > 0 else "run export first")

## 10. Metadata: one row per film from TMDb + IMDb (+ TMDb credits)

`credits` is a second API pass (`/movie/{id}/credits`, ~4 h for the catalogue) — run it
from a terminal like the first fetch. `metadata` is offline: it joins the stored TMDb
payload, the stored credits, and `imdb_match/imdb.db` (title type, year, runtime,
genres, rating, votes, directors, writers, top-billed cast) into `film_metadata`, and
`export` writes it as `films.csv`. Sources that are absent for a film are recorded as
`missing`, never as blanks that look like "no director".

In [ ]:
if cfg.live and RUN_FULL_FETCH:
    R.fetch_stage(cfg, con, kind="credits")
elif cfg.live:
    R.fetch_stage(cfg, con, kind="credits", limit=300)
R.metadata_stage(cfg, con)
pd.read_sql("SELECT tmdb_id, tconst, title, year_imdb, genres_tmdb, directors_imdb, directors_tmdb, cast_imdb, "
            "imdb_rating, vote_average, imdb_source, credits_source FROM film_metadata LIMIT 8", con)

## 11. Box Office Mojo

`box_office.py` shares this database. `scope` picks every film with a TMDb theatrical
entry (type 2/3) dated 2000 or later; `fetch` pulls one page at a time with a pause and
caches everything (most films have no BOM page and cost one request; films with data cost
3–5); `parse` is offline and re-runnable; `match` joins BOM territory rows to the TMDb
theatrical events by ISO code and date. Run the long fetch from a terminal:

```bash
python3 box_office.py scope
python3 box_office.py fetch --limit 200     # read it, then
python3 box_office.py fetch                 # hours to days, resumable
python3 box_office.py parse && python3 box_office.py match && python3 box_office.py export
```

In [ ]:
import box_office as B
bcon = B.connect(cfg)
B.scope_stage(cfg, bcon)
B.fetch_stage(cfg, bcon, limit=25)
B.parse_stage(cfg, bcon); B.match_stage(cfg, bcon); B.export_stage(cfg, bcon)
B.report(cfg, bcon)
pd.read_sql("SELECT s.tconst, f.title, s.distributor, s.domestic_usd, s.international_usd, s.worldwide_usd, "
            "s.domestic_opening_usd, s.opening_theaters, s.widest_release, s.domestic_release_date, s.uk_distributor, s.uk_gross_usd "
            "FROM bo_summary s JOIN films f USING(tmdb_id) ORDER BY s.worldwide_usd DESC LIMIT 10", bcon)

## 12. Enrichment: awards, ratings, distributors

`enrich.py` shares the database. `wikidata` is cheap (batched SPARQL by tconst) and is
what gives the other two scrapes their Rotten Tomatoes / Metacritic ids. `omdb` is
budgeted per run (free tier: 1,000/day). `rt` and `tmdb_awards` are long scrapes —
run them from a terminal with `nohup`; the cells below only do a handful.

In [ ]:
import enrich as E
econ = E.connect(cfg)
E.wikidata_stage(cfg, econ, limit=250)
if os.environ.get("OMDB_API_KEY"):
    E.omdb_stage(cfg, econ, limit=25)
E.export_stage(cfg, econ)
E.report(cfg, econ)
pd.read_sql("SELECT a.tconst, f.title, a.kind, a.award_label, a.when_date, a.family, a.fest_qid FROM wd_awards a "
            "JOIN films f USING(tmdb_id) ORDER BY a.when_date DESC LIMIT 15", econ)

## 13. The type-1 question, measured

The rebuild plan's first hypothesis was that filtering on `type == 1` drops
later festivals in the same country. On the two real payloads it drops **none**
— contributors entered every festival as type 1. Whether that holds across the
catalogue is what the number below answers, on real data, once the full fetch
has run.

In [ ]:
n_app, lost = con.execute("SELECT COUNT(*), SUM(min_type != 1) FROM appearances").fetchone()
print(f"{n_app} festival appearances; a type==1 filter would lose {lost or 0} ({100*(lost or 0)/max(n_app,1):.1f}%)")
pd.read_sql("SELECT min_type, COUNT(*) n FROM appearances GROUP BY min_type", con)